In [ ]:
# installing the Kaggle library
!pip install kaggle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# configuring the path of Kaggle.json file
!mkdir -p ~/.kaggle
!cp 'kaggle (3).json' ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
! kaggle competitions download -c dogs-vs-cats

dogs-vs-cats.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
# Extracting the compressed dataset
from zipfile import ZipFile
dataset = '/content/dogs-vs-cats.zip'
with ZipFile(dataset, 'r') as zip:
  zip.extractall()
  print('The dataset is extracted')

The dataset is extracted


The `dogs-vs-cats.zip` file contains `train.zip` and `test1.zip`. We need to extract `train.zip` to get the actual `cat` and `dog` images. The `ImageDataGenerator` will then point to the extracted `train` directory.

In [ ]:
import os
from zipfile import ZipFile

# Path to the train.zip file, which is inside the previously extracted dogs-vs-cats.zip
train_zip_path = '/content/train.zip'
# Directory where the contents of train.zip will be extracted
extract_train_dir = '/content/extracted_train'

# Create the directory if it doesn't exist
if not os.path.exists(extract_train_dir):
    os.makedirs(extract_train_dir)

# Extract the train.zip file
with ZipFile(train_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_train_dir)
print(f'The train dataset (containing cat and dog folders) is extracted to {extract_train_dir}/train')


The train dataset (containing cat and dog folders) is extracted to /content/extracted_train/train


The `ImageDataGenerator.flow_from_directory` expects images to be organized into subdirectories named after their classes. Since `train.zip` extracts images directly into the `train` folder, we need to create 'cat' and 'dog' subdirectories and move the respective images into them.

In [ ]:
import shutil
import os

source_dir = '/content/extracted_train/train'

# Create 'cat' and 'dog' subdirectories
cat_dir = os.path.join(source_dir, 'cat')
dog_dir = os.path.join(source_dir, 'dog')

if not os.path.exists(cat_dir):
    os.makedirs(cat_dir)
if not os.path.exists(dog_dir):
    os.makedirs(dog_dir)

# Move images to their respective class subdirectories
for filename in os.listdir(source_dir):
    # Ensure we only process files and not the newly created directories
    file_path = os.path.join(source_dir, filename)
    if os.path.isfile(file_path):
        if filename.startswith('cat'):
            shutil.move(file_path, os.path.join(cat_dir, filename))
        elif filename.startswith('dog'):
            shutil.move(file_path, os.path.join(dog_dir, filename))

print(f"Images moved to {cat_dir} and {dog_dir}.")

Images moved to /content/extracted_train/train/cat and /content/extracted_train/train/dog.


**Import Libraries**

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
import numpy as np

**Data Preprocessing (Data ko ready karna)**

In [ ]:
# ImageDataGenerator images ko read karta hai, unhe 0-1 ki range mein lata hai (rescale)
# aur data badhane ke liye unhe thora ghumata aur flip karta hai (Data Augmentation)

In [ ]:
datagen = ImageDataGenerator(
    rescale=1./255,         # Pixels ko 0 se 1 ke darmiyan set karna
    shear_range=0.2,        # Image ko thora sa tehra karna
    zoom_range=0.2,         # Image ko zoom karna
    horizontal_flip=True,   # Image ko mirror ki tarah palatna
    validation_split=0.2    # 80% data training ke liye, 20% testing/validation ke liye
)

**Load Training Data**

In [ ]:
train_generator = datagen.flow_from_directory(
    '/content/dataset',
    target_size=(128, 128),      # Har image ko 128x128 size mein convert karna
    batch_size=32,               # Ek waqt mein 32 images process hongi
    class_mode='binary',         # Binary kyunke sirf 2 classes hain (Cat ya Dog)
    subset='training'
)

Found 40000 images belonging to 3 classes.


In [ ]:
train_generator = datagen.flow_from_directory(
    '/content/extracted_train/train', # Corrected path to the directory containing 'cat' and 'dog' folders
    target_size=(128, 128),      # Har image ko 128x128 size mein convert karna
    batch_size=32,               # Ek waqt mein 32 images process hongi
    class_mode='binary',         # Binary kyunke sirf 2 classes hain (Cat ya Dog)
    subset='training'
)

Found 20000 images belonging to 2 classes.


**Load Validation Data**

In [ ]:
validation_generator = datagen.flow_from_directory(
    '/content/dataset',
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

Found 10000 images belonging to 3 classes.


In [ ]:
validation_generator = datagen.flow_from_directory(
    '/content/extracted_train/train', # Corrected path to the directory containing 'cat' and 'dog' folders
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

Found 5000 images belonging to 2 classes.


**Building the CNN Model**

In [ ]:
model = Sequential()

# 1st Convolutional Layer (Features nikalne ke liye)
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)))
model.add(MaxPooling2D(pool_size=(2, 2))) # Image ka size chota karne ke liye

# 2nd Convolutional Layer
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# 3rd Convolutional Layer
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Flatten Layer (2D data ko 1D mein convert karna)
model.add(Flatten())

# Fully Connected Layer (Dimagh ki tarah final decision lene wali layer)
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5)) # Overfitting (ratta lagane) se bachane ke liye

# Output Layer (0 = Cat, 1 = Dog)
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


**Compiling the Model**

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

**Training the Model (Model ko sikhana)**

In [ ]:
#epochs 10 mean model 10 dfa pora data ko dekhy ga
model.fit(train_generator,validation_data=validation_generator,epochs=10)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 139s 215ms/step - accuracy: 0.6162 - loss: 0.6445 - val_accuracy: 0.7030 - val_loss: 0.5603
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 132s 211ms/step - accuracy: 0.7268 - loss: 0.5402 - val_accuracy: 0.7568 - val_loss: 0.4991
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 147s 235ms/step - accuracy: 0.7714 - loss: 0.4825 - val_accuracy: 0.7980 - val_loss: 0.4333
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 128s 205ms/step - accuracy: 0.7956 - loss: 0.4434 - val_accuracy: 0.8086 - val_loss: 0.4220
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 144s 231ms/step - accuracy: 0.8109 - loss: 0.4182 - val_accuracy: 0.8162 - val_loss: 0.3967
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 130s 208ms/step - accuracy: 0.8304 - loss: 0.3815 - val_accuracy: 0.8300 - val_loss: 0.3830
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 132s 211ms/step - accuracy: 0.8432 - loss: 0.3545 - val_accuracy: 0.8412 - val_loss: 0.3488
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 135s 216ms/step - accuracy: 0.8526 -

**Save the model**

In [ ]:
# Save the model so that we don't have to train it again and again

In [ ]:
model.save('cat_dog_model.h5')

**Testing on a New Image (Nayi image par check karna)**

In [ ]:
def predict_image(img_path):
    # Image ko load aur resize karna
    img = image.load_img(img_path, target_size=(128, 128))

    # Image ko array (numbers) mein convert karna
    img_array = image.img_to_array(img)

    # Keras hamesha batches mein kaam karta hai, isliye dimensions expand karni parti hain
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0 # Rescale

    # Prediction karna
    prediction = model.predict(img_array)

    # Agar answer 0.5 se bara hai to Dog, warna Cat
    if prediction[0][0] > 0.5:
      return "Ye ek Dog 🐶 hai!"
    else:
        return "Ye ek Cat 🐱 hai!"

In [ ]:
# Put the path of any new picture here and check

In [ ]:
print(predict_image('/content/Cat_November_2010-1a.jpg'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Ye ek Cat 🐱 hai!


In [ ]:
print(predict_image('/content/test-dog-jpg.webp'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Ye ek Dog 🐶 hai!
